In [1]:
from dotenv import load_dotenv
load_dotenv()
from openai import OpenAI
openai_client = OpenAI()

In [2]:
def llm(prompt):
    response = openai_client.responses.create(
        model="gpt-5.4-mini",
        input=prompt
    )
    return response.output_text

In [ ]:
llm('Hey How are you?')

'If you have a **₹1 crore loan** at **7.3% annual interest** and can pay **₹2 lakh per month**, the key question is whether that **₹2 lakh is enough to cover the interest + reduce principal**.\n\n### Quick estimate\n- **Monthly interest rate** = 7.3% / 12 ≈ **0.608%**\n- **Interest in first month on ₹1 crore** = about **₹60,833**\n- If you pay **₹2,00,000/month**, then about:\n  - **₹60,833** goes to interest\n  - **₹1,39,167** goes to principal\n\nSo yes, **₹2 lakh/month is a strong repayment amount** and will reduce your loan steadily.\n\n### Approximate payoff time\nWith:\n- Loan = **₹1,00,00,000**\n- Interest = **7.3%**\n- EMI/payment = **₹2,00,000 per month**\n\nYou would likely finish the loan in about **6.5 to 7 years**.\n\n### Best way to repay\n1. **Pay as early as possible each month**\n   - Interest is usually calculated on the outstanding balance, so early payment can help a little.\n\n2. **Keep paying more than minimum EMI**\n   - Since ₹2 lakh is likely higher than the re

In [3]:
def rag(question):
    serach_results = search(question)
    user_prompt = build_prompt(question, serach_results)
    return llm(user_prompt)

In [4]:
import requests

docs_url = "https://datatalks.club/faq/json/courses.json"
response = requests.get(docs_url)
courses_raw = response.json()

In [5]:
documents = []
url_prefix = "https://datatalks.club/faq"

for course in courses_raw:
    course_url = f"""{url_prefix}{course["path"]}"""

    course_response = requests.get(course_url)
    course_response.raise_for_status()
    course_data = course_response.json()

    documents.extend(course_data)

len(documents)

1342

In [6]:
from minsearch import Index

index = Index(
    text_fields=["question", "section", "answer"],
    keyword_fields=["course"]
)

index.fit(documents)

In [7]:
question = "I just discovered the course can I Join it"

In [8]:
search_results = index.search(
    question,
    boost_dict={'question':2.0},
    filter_dict = {'course':'llm-zoomcamp'},
    num_results=5)

In [9]:
def search(question, course="llm-zoomcamp"):
    boost_dict = {"question": 2.0, "section": 0.5}
    filter_dict = {"course": course}

    return index.search(
        question,
        boost_dict=boost_dict,
        filter_dict=filter_dict,
        num_results=5
    )

In [10]:
search_results = search(question)

In [11]:
INSTRUCTIONS = """
Your task is to answer questions from the course participants
based on the provided context.

Use the context to find relevant information and provide accurate
answers. If the answer is not found in the context,
respond with "I don't know."
"""

In [12]:
USER_PROMPT_TEMPLATE = """
Question:
{question}

Context:
{context}
"""

In [13]:
def build_context(search_results):
    lines = []

    for doc in search_results:
        lines.append(doc["section"])
        lines.append("Q: " + doc["question"])
        lines.append("A: " + doc["answer"])
        lines.append("")

    return "\n".join(lines).strip()

In [14]:
def build_prompt(question, search_results):
    context = build_context(search_results)
    prompt = USER_PROMPT_TEMPLATE.format(
        question=question,
        context=context
    )
    return prompt.strip()

In [26]:
prompt = build_prompt(question, search_results)

In [27]:
print(prompt)

Question:
I just discovered the course can I Join it

Context:
General Course-Related Questions
Q: I just discovered the course. Can I still join?
A: Yes, but if you want to receive a certificate, you need to submit your project while we’re still accepting submissions.

General Course-Related Questions
Q: Course: I have registered for the LLM Zoomcamp. When can I expect to receive the confirmation email?
A: You don't need it. You're accepted. You can also just start learning and submitting homework (while the form is open) without registering. It is not checked against any registered list. Registration is just to gauge interest before the start date.

General Course-Related Questions
Q: Certificate: Can I follow the course in a self-paced mode and get a certificate?
A: No, you can only get a certificate if you finish the course with a "live" cohort.

We don't award certificates for the self-paced mode. The reason is you need to peer-review 3 capstone(s) after submitting your project.



In [ ]:
response = openai_client.responses.create(
    model = 'gpt-5.4-mini',
    input = prompt
)

In [21]:
response.output_text

'Yes — you can still join and start learning.\n\nIf you want a certificate, make sure to submit your project while submissions are still open. If the course is no longer running live, you can still follow it, but certificates are only awarded to participants in the live cohort.'

In [23]:
print(response.model_dump_json(indent=2))

{
  "id": "resp_03704a9c30f6fb9c006a2ec6cc21288191bffb9bdf30971cfd",
  "created_at": 1781450444.0,
  "error": null,
  "incomplete_details": null,
  "instructions": null,
  "metadata": {},
  "model": "gpt-5.4-mini-2026-03-17",
  "object": "response",
  "output": [
    {
      "id": "msg_03704a9c30f6fb9c006a2ec6ccf584819195a324ad4bd72204",
      "content": [
        {
          "annotations": [],
          "text": "Yes — you can still join and start learning.\n\nIf you want a certificate, make sure to submit your project while submissions are still open. If the course is no longer running live, you can still follow it, but certificates are only awarded to participants in the live cohort.",
          "type": "output_text",
          "logprobs": []
        }
      ],
      "role": "assistant",
      "status": "completed",
      "type": "message",
      "phase": "final_answer"
    }
  ],
  "parallel_tool_calls": true,
  "temperature": 1.0,
  "tool_choice": "auto",
  "tools": [],
  "top_p": 

In [24]:
input_price = 0.75 / 1_000_000
output_price = 4.50 / 1_000_000

cost = (
    response.usage.input_tokens * input_price +
    response.usage.output_tokens * output_price
)

cost

0.0005152500000000001

#### Message history

In [28]:
message_history = [
    {"role": "developer", "content": INSTRUCTIONS},
    {"role": "user", "content": prompt}
]

response = openai_client.responses.create(
    model="gpt-5.4-mini",
    input=message_history
)

#### The LLM function

In [29]:
def llm(instructions, user_prompt, model="gpt-5.4-mini"):
    message_history = [
        {"role": "developer", "content": instructions},
        {"role": "user", "content": user_prompt}
    ]

    response = openai_client.responses.create(
        model=model,
        input=message_history
    )

    return response.output_text

#### Full RAG

In [30]:
def rag(query, model="gpt-5.4-mini"):
    search_results = search(query)
    prompt = build_prompt(query, search_results)
    answer = llm(INSTRUCTIONS, prompt, model=model)
    return answer

In [32]:
answer = rag("I just discovered the course. Can I join now?")
print(answer)

Yes, you can still join and start learning. If you want a certificate, you need to submit your project while submissions are still being accepted.
